In [1]:
import requests
from bs4 import BeautifulSoup

Stopwords are used when building the inverted index. The inverted index will ignore stopwords.

In [2]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

STOPWORDS = stopwords.words('english')
print(STOPWORDS)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Add custom stopwords if you deem it necessary

In [3]:
custom_STOPWORDS = [] # Add your own stopwords here
STOPWORDS.extend(custom_STOPWORDS)

In [4]:
from collections import defaultdict

# Inverted index: word -> set of URLs
inverted_index = defaultdict(set)
url_list = set()

In [5]:
# This dictionary will be used to build the connection between links
web_connection = {'source':[], 'target':[]}

In [6]:
import re

# This function will clean the content of web page in order to build the inverted index.
def clean_and_tokenize(text):
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text.lower())  # Remove punctuation and lowercase
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

In [7]:
from urllib.parse import urljoin, urlparse

# The crawl function has 5 parameters
# url = The url to crawl
# base_domain = the base domain of the url. During crawling, the crawler will ignore links from other domains

def crawl(url, base_domain, visited, visit_limit, limit):
    if limit==0 or len(visited)==visit_limit:
        return

    try:
        response = requests.get(url, timeout=5)
        if response.status_code != 200:
            return
    except requests.RequestException:
        return

    visited.add(url)
    print("-"*(10-limit), end=" ")
    print(f"Crawled: {url}")

    soup = BeautifulSoup(response.text, 'html.parser')
    text = soup.get_text(separator=' ', strip=True)
    words = clean_and_tokenize(text)

    for word in words:
        inverted_index[word].add(url)
        url_list.add(url)

    # Recursively follow links
    for tag in soup.find_all('a', href=True):
        link = urljoin(url, tag['href'])
        parsed = urlparse(link)

        # Store external links as connection
        web_connection['source'].append(url)
        web_connection['target'].append(link)

        if parsed.netloc == base_domain and link not in visited:
            crawl(link, base_domain, visited, visit_limit, limit-1)

In [8]:
def crawl_roots(root_urls, max_per_root=2, visit_limit=50):
    for root in root_urls:
        print(f"\nStarting crawl from: {root}")
        domain = urlparse(root).netloc
        visited = set()
        crawl(root, domain, visited, visit_limit, max_per_root)

In [13]:
seed_urls = [
    'https://www.geeksforgeeks.org/'
]

crawl_roots(seed_urls, max_per_root=10)


Starting crawl from: https://www.geeksforgeeks.org/
 Crawled: https://www.geeksforgeeks.org/
- Crawled: https://www.geeksforgeeks.org/dsa/dsa-tutorial-learn-data-structures-and-algorithms/
-- Crawled: https://www.geeksforgeeks.org/dsa/top-100-data-structure-and-algorithms-dsa-interview-questions-topic-wise/
--- Crawled: https://www.geeksforgeeks.org/dsa/data-structures-and-algorithms-online-quiz/
---- Crawled: https://www.geeksforgeeks.org/dsa/must-do-coding-questions-for-companies-like-amazon-microsoft-adobe/
----- Crawled: https://www.geeksforgeeks.org/dsa/advanced-data-structures/
------ Crawled: https://www.geeksforgeeks.org/system-design/system-design-tutorial/
------- Crawled: https://www.geeksforgeeks.org/system-design/what-is-high-level-design-learn-system-design/
-------- Crawled: https://www.geeksforgeeks.org/system-design/what-is-low-level-design-or-lld-learn-system-design/
--------- Crawled: https://www.geeksforgeeks.org/software-engineering/functional-vs-non-functional-re

In [15]:
# print inverted index
print("\nSample inverted index (first 20 words):")
for word in list(inverted_index.keys())[:50]:
    print(f"{word}: {list(inverted_index[word])}")


Sample inverted index (first 20 words):
bbc: ['https://www.bbc.com/health', 'https://www.bbc.com/technology/ai-v-the-mind', 'https://www.bbc.com/live', 'https://www.bbc.com/news/bbcverify', 'https://www.bbc.com/news/world/asia/india', 'https://www.bbc.com/news/world/europe', 'https://www.bbc.com/culture/music', 'https://www.theguardian.com/music/classicalmusicandopera', 'https://www.bbc.com/arts', 'https://www.bbc.com/news/northern_ireland', 'https://www.bbc.com/business', 'https://www.bbc.com/sport', 'https://www.bbc.com/news/world/middle_east', 'https://www.bbc.com/culture/books', 'https://www.bbc.com/business/technology-of-business', 'https://www.bbc.com/#main-content', 'https://www.bbc.com', 'https://www.bbc.com/travel', 'https://www.bbc.com/news/world/asia', 'https://www.bbc.com/technology', 'https://www.bbc.com/news/world/asia/china', 'https://www.bbc.com/news/scotland', 'https://www.bbc.com/culture/art', 'https://www.bbc.com/news/wales/wales_politics', 'https://www.bbc.com/news

In [17]:
# Print first 20 connections

for source, target in list(zip(web_connection['source'], web_connection['target']))[:100]:
    print(f"{source} -> {target}")

https://www.bbc.com/news -> https://www.bbc.com/news#main-content
https://www.bbc.com/news#main-content -> https://www.bbc.com/news#main-content
https://www.bbc.com/news#main-content -> https://www.bbc.com/
https://www.bbc.com/ -> https://www.bbc.com/#main-content
https://www.bbc.com/#main-content -> https://www.bbc.com/#main-content
https://www.bbc.com/#main-content -> https://www.bbc.com/
https://www.bbc.com/#main-content -> https://www.bbc.com/
https://www.bbc.com/#main-content -> https://www.bbc.com/news
https://www.bbc.com/#main-content -> https://www.bbc.com/sport
https://www.bbc.com/sport -> https://www.bbc.com
https://www.bbc.com -> https://www.bbc.com#main-content
https://www.bbc.com#main-content -> https://www.bbc.com#main-content
https://www.bbc.com#main-content -> https://www.bbc.com/
https://www.bbc.com#main-content -> https://www.bbc.com/
https://www.bbc.com#main-content -> https://www.bbc.com/news
https://www.bbc.com#main-content -> https://www.bbc.com/sport
https://www.